In [ ]:
#!pip install tqdm
#!pip install statsmodels

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
from joblib import Parallel, delayed
from tqdm.auto import tqdm
from statsmodels.stats.multitest import multipletests

/home/dcm/env/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ------------------------------------------------------------
# Load your tables (modify paths or replace with in-memory objects)
# ------------------------------------------------------------
otu = pd.read_csv("pgfam/pgfam_otu.txt", sep = '\t')
otu = otu.melt(id_vars = 'pgfam', var_name = 'Metagenomic_file_name', value_name = 'count')
otu['Metagenomic_file_name'] = 'X' + otu['Metagenomic_file_name']
otu['Metagenomic_file_name'] = otu['Metagenomic_file_name'].str.replace('-', '.')
otu

,pgfam,Metagenomic_file_name,count
0,PGF_00000000,X250513Pat_D25.7849,0
1,PGF_00000017,X250513Pat_D25.7849,0
2,PGF_00000020,X250513Pat_D25.7849,0
3,PGF_00000023,X250513Pat_D25.7849,0
4,PGF_00000025,X250513Pat_D25.7849,0
...,...,...,...
7364299,PGF_12958850,X250930Pat_D25.12478,0
7364300,PGF_12958902,X250930Pat_D25.12478,1
7364301,PGF_12958914,X250930Pat_D25.12478,0
7364302,PGF_12958983,X250930Pat_D25.12478,0


In [3]:
metadata = pd.read_csv("pgfam/mgx_metadata.txt", sep = '\t')

In [4]:
dfm = pd.merge(otu, metadata, on = 'Metagenomic_file_name', how = 'left')
dfm

,pgfam,Metagenomic_file_name,count,Biopsy_collection_date_year,Batch,Shipment date,Sex,Age at baseline,Prog-Nonprog between 20-26y,Dx_20y,...,Total Isolates,Unnamed: 23,Tube Sample ID - Culture Plate DNA,Date and Time,Nucleic Acid ng/µl,260/280,260/230,Unnamed: 29,Metagenomics_batch,Metagenomic_BMC_ID
0,PGF_00000000,X250513Pat_D25.7849,0,Year_20,1,11/6/2023,M,54,N,IM,...,23.0,NaN,68,3/29/2025 14:41,430.9,1.90,2.18,NaN,1,D25-7849
1,PGF_00000017,X250513Pat_D25.7849,0,Year_20,1,11/6/2023,M,54,N,IM,...,23.0,NaN,68,3/29/2025 14:41,430.9,1.90,2.18,NaN,1,D25-7849
2,PGF_00000020,X250513Pat_D25.7849,0,Year_20,1,11/6/2023,M,54,N,IM,...,23.0,NaN,68,3/29/2025 14:41,430.9,1.90,2.18,NaN,1,D25-7849
3,PGF_00000023,X250513Pat_D25.7849,0,Year_20,1,11/6/2023,M,54,N,IM,...,23.0,NaN,68,3/29/2025 14:41,430.9,1.90,2.18,NaN,1,D25-7849
4,PGF_00000025,X250513Pat_D25.7849,0,Year_20,1,11/6/2023,M,54,N,IM,...,23.0,NaN,68,3/29/2025 14:41,430.9,1.90,2.18,NaN,1,D25-7849
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7364299,PGF_12958850,X250930Pat_D25.12478,0,Year_20,3,6/10/2024,M,31,P,IM,...,NaN,NaN,116,9/27/2025 13:26,440.5,1.91,2.16,NaN,3,D25-12478
7364300,PGF_12958902,X250930Pat_D25.12478,1,Year_20,3,6/10/2024,M,31,P,IM,...,NaN,NaN,116,9/27/2025 13:26,440.5,1.91,2.16,NaN,3,D25-12478
7364301,PGF_12958914,X250930Pat_D25.12478,0,Year_20,3,6/10/2024,M,31,P,IM,...,NaN,NaN,116,9/27/2025 13:26,440.5,1.91,2.16,NaN,3,D25-12478
7364302,PGF_12958983,X250930Pat_D25.12478,0,Year_20,3,6/10/2024,M,31,P,IM,...,NaN,NaN,116,9/27/2025 13:26,440.5,1.91,2.16,NaN,3,D25-12478


In [5]:
dfm["pa"] = (dfm["count"] > 0).astype(int)
dfm

,pgfam,Metagenomic_file_name,count,Biopsy_collection_date_year,Batch,Shipment date,Sex,Age at baseline,Prog-Nonprog between 20-26y,Dx_20y,...,Unnamed: 23,Tube Sample ID - Culture Plate DNA,Date and Time,Nucleic Acid ng/µl,260/280,260/230,Unnamed: 29,Metagenomics_batch,Metagenomic_BMC_ID,pa
0,PGF_00000000,X250513Pat_D25.7849,0,Year_20,1,11/6/2023,M,54,N,IM,...,NaN,68,3/29/2025 14:41,430.9,1.90,2.18,NaN,1,D25-7849,0
1,PGF_00000017,X250513Pat_D25.7849,0,Year_20,1,11/6/2023,M,54,N,IM,...,NaN,68,3/29/2025 14:41,430.9,1.90,2.18,NaN,1,D25-7849,0
2,PGF_00000020,X250513Pat_D25.7849,0,Year_20,1,11/6/2023,M,54,N,IM,...,NaN,68,3/29/2025 14:41,430.9,1.90,2.18,NaN,1,D25-7849,0
3,PGF_00000023,X250513Pat_D25.7849,0,Year_20,1,11/6/2023,M,54,N,IM,...,NaN,68,3/29/2025 14:41,430.9,1.90,2.18,NaN,1,D25-7849,0
4,PGF_00000025,X250513Pat_D25.7849,0,Year_20,1,11/6/2023,M,54,N,IM,...,NaN,68,3/29/2025 14:41,430.9,1.90,2.18,NaN,1,D25-7849,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7364299,PGF_12958850,X250930Pat_D25.12478,0,Year_20,3,6/10/2024,M,31,P,IM,...,NaN,116,9/27/2025 13:26,440.5,1.91,2.16,NaN,3,D25-12478,0
7364300,PGF_12958902,X250930Pat_D25.12478,1,Year_20,3,6/10/2024,M,31,P,IM,...,NaN,116,9/27/2025 13:26,440.5,1.91,2.16,NaN,3,D25-12478,1
7364301,PGF_12958914,X250930Pat_D25.12478,0,Year_20,3,6/10/2024,M,31,P,IM,...,NaN,116,9/27/2025 13:26,440.5,1.91,2.16,NaN,3,D25-12478,0
7364302,PGF_12958983,X250930Pat_D25.12478,0,Year_20,3,6/10/2024,M,31,P,IM,...,NaN,116,9/27/2025 13:26,440.5,1.91,2.16,NaN,3,D25-12478,0


In [6]:
otu_pa = dfm.pivot_table(
    index="Metagenomic_file_name",
    columns="pgfam",
    values="pa",
    fill_value=0
)

In [7]:
metadata = dfm[["Metagenomic_file_name", "Biopsy_collection_date_year"]].drop_duplicates()
metadata = metadata.set_index("Metagenomic_file_name")

In [8]:
P_ids = metadata[metadata["Biopsy_collection_date_year"] == "Year_20"].index
N_ids = metadata[metadata["Biopsy_collection_date_year"] == "Year_26"].index

otu_P = otu_pa.loc[P_ids]
otu_N = otu_pa.loc[N_ids]

In [9]:
def run_fisher(otu):
    p_vec = otu_P[otu].values
    n_vec = otu_N[otu].values

    # 2x2 contingency table
    contingency = np.array([
        [p_vec.sum(), len(p_vec) - p_vec.sum()],
        [n_vec.sum(), len(n_vec) - n_vec.sum()]
    ])

    # Skip degenerate tables
    if np.all(contingency == 0):
        return None
    if (contingency.sum(axis=0) == 0).any() or (contingency.sum(axis=1) == 0).any():
        return None

    try:
        odds_ratio, p_value = fisher_exact(contingency)
    except Exception:
        return None

    return {"otu": otu, "Odds_Ratio": odds_ratio, "P": p_value}


In [10]:
# ------------------------------------------------------------
# Parallel Fisher tests with progress bar
# ------------------------------------------------------------
otus = otu_pa.columns

results = Parallel(n_jobs=-1, backend="loky")(
    delayed(run_fisher)(otu) 
    for otu in tqdm(otus, desc="Running Fisher tests", ncols=80)
)

# Remove None
results = [r for r in results if r is not None]

results_df = pd.DataFrame(results)

Running Fisher tests: 100%|█████████████| 102282/102282 [28:21<00:00, 60.11it/s]


In [11]:
# ------------------------------------------------------------
# FDR correction
# ------------------------------------------------------------
results_df["FDR_BH"] = multipletests(results_df["P"], method="fdr_bh")[1]

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------
results_df.to_csv("fisher_20_vs_26_results.txt", sep="\t", index=False)

In [ ]:
results_df